# Проект: проверка гипотезы в Python и составление аналитической записки

- Автор: Ксенофонтов Никита
- Дата: 08.09.2025

## Цели и задачи проекта

Проведем проверку гипотез в сервисе Книги и выясним, проводят ли в среднем больше времени за чтением и прослушиванием книг в приложении пользователи из Санкт-Петербурга, чем пользователи из Москвы. Для этого предварительно загрузим данные пользователей и проведем предобработку.

## Описание данных

`knigi_data.csv` — таблица с данными пользователей Яндекс.Книги.

Поля таблицы:
- `city` — название города.
- `puid` — id пользователя.
- `hours` — суммарное кол-во часов активности пользователя.

## Содержимое проекта

1. Загрузка данных и знакомство с ними
2. Проверка гипотезы в Python
3. Аналитическая записка

---

## Загрузка данных и знакомство с ними

Загрузим данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания) из файла `/datasets/knigi_data.csv`.

In [1]:
#Импортируем библиотеки
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.stats import ttest_ind
    from statsmodels.stats.power import NormalIndPower
    from statsmodels.stats.proportion import proportion_effectsize
    from statsmodels.stats.proportion import proportions_ztest
except ImportError as e:
    print(f'Ошибка при импорте библиотеки: {e}') 

In [1]:
#Создадим датасет df_knigi
df_knigi = pd.read_csv('/datasets/yandex_knigi_data.csv')

NameError: name 'pd' is not defined

In [3]:
#Выведем информацию о датасете
df_knigi.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  8784 non-null   int64  
 1   city        8784 non-null   object 
 2   puid        8784 non-null   int64  
 3   hours       8784 non-null   float64
dtypes: float64(1), int64(2), object(1)
memory usage: 274.6+ KB


In [4]:
#Выведем несколько строк датасета для ознакомления
df_knigi.head()

,Unnamed: 0,city,puid,hours
0,0,Москва,9668,26.167776
1,1,Москва,16598,82.111217
2,2,Москва,80401,4.656906
3,3,Москва,140205,1.840556
4,4,Москва,248755,151.326434


Можем заметить лишний столбец Unnamed:0, удалим его, чтобы не мешал

In [5]:
df_knigi.drop('Unnamed: 0', axis=1, inplace=True)

In [6]:
#Проверим данные на полные дубликаты
number_of_duplicates = df_knigi.duplicated().sum()
print(f"Количество полных дубликатов в датасете: {number_of_duplicates}")

Количество полных дубликатов в датасете: 0


In [7]:
df_knigi['hours'].describe()

count    8784.000000
mean       11.087670
std        37.701350
min         0.000018
25%         0.066246
50%         0.942344
75%         6.065151
max       978.764775
Name: hours, dtype: float64

В датасете присутствуют большие выбросы, отсеем значения до 99ого процентиля

In [8]:
#С помощью фильтра отсеем выбросы
df_knigi = df_knigi[df_knigi['hours'] <= df_knigi['hours'].quantile(0.99)]

## Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуем статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

In [9]:
#Разделим пользователей по спискам
A_list = df_knigi[df_knigi['city'] == 'Москва']['puid']
B_list = df_knigi[df_knigi['city'] == 'Санкт-Петербург']['puid']
#Найдем пересечения в списках
intersection = list(set(A_list) & set(B_list))
#Выведем результат
len(intersection)

237

In [10]:
#Разделим датасет на группы и исключим пересекающиеся строки
A_df = df_knigi[(df_knigi['city'] == 'Москва') & (~df_knigi['puid'].isin(intersection))].hours
B_df = df_knigi[(df_knigi['city'] == 'Санкт-Петербург') & (~df_knigi['puid'].isin(intersection))].hours

In [11]:
A_df.describe()

count    5934.000000
mean        8.089960
std        19.816739
min         0.000022
25%         0.055570
50%         0.859626
75%         5.651015
max       153.966600
Name: hours, dtype: float64

In [12]:
B_df.describe()

count    2288.000000
mean        8.438326
std        19.950322
min         0.000025
25%         0.057674
50%         0.856364
75%         5.707955
max       156.882789
Name: hours, dtype: float64

Размер выборок оказался очень разным, 5934 жителей Москвы и 2288 жителей Санкт-Петербурга, к тому же мы не знаем равны ли выборочные дисперсии, поэтому для большей надежности будем использовать т-тест Уэлча и установим уровень значимости a = 0.05, как стандартный уровень для сравнения продуктовых метрик.

In [13]:
#Применим тест Уэлча
test_result = ttest_ind(A_df, B_df, equal_var=False, alternative='less')
alpha = 0.05

print(test_result.pvalue)

if test_result.pvalue < alpha:
    print('Отвергаем нулевую гипотезу')
else:
    print('Не получилось отвергнуть нулевую гипотезу')

0.2385941087347347
Не получилось отвергнуть нулевую гипотезу


## Аналитическая записка
По результатам анализа данных подготовим аналитическую записку


Так как объёмы выборок различаются и дисперсии неизвестны, применён t-тест Уэлча для двух независимых выборок. Уровень статистической значимости `a` = 0.05.

Было получено значение `p-value` = 0.2385941087347347, что больше `a` и говорит нам о том, что пользователи Санкт-Петербурга не проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы.

Можно предположить, что пользователи обоих городов в целом схожи по образу жизни и шаблонам чтения, либо влияние других факторов (возраст, профессия, сезонность) сильнее географической привязки.

----

# Часть 2. Анализ результатов A/B-тестирования

Теперь нам нужно проанализировать другие данные. К нам обратились представители интернет-магазина BitMotion Kit, в котором продаются геймифицированные товары для тех, кто ведёт здоровый образ жизни. У него есть своя целевая аудитория, даже появились хиты продаж: эспандер со счётчиком и напоминанием, так и подстольный велотренажёр с Bluetooth.

В будущем компания хочет расширить ассортимент товаров. Но перед этим нужно решить одну проблему. Интерфейс онлайн-магазина слишком сложен для пользователей — об этом говорят отзывы.

Чтобы привлечь новых клиентов и увеличить число продаж, владельцы магазина разработали новую версию сайта и протестировали его на части пользователей. По задумке, это решение доказуемо повысит количество пользователей, которые совершат покупку.

Наша задача — провести оценку результатов A/B-теста. В Нашем распоряжении:

* данные о действиях пользователей и распределении их на группы,

* техническое задание.

Оценим корректность проведения теста и проанализируем его результаты.

## Опишем цели исследования.



Цели исследования:

Проверить, увеличивает ли новая версия сайта долю пользователей, совершивших покупку (конверсию) по сравнению с текущим интерфейсом. Убедиться в корректности проведения A/B-теста. На основе полученных результатов принять обоснованное решение о полном развёртывании нового интерфейса.

## Загрузим данные, оценим их целостность.


In [14]:
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

In [15]:
#Выведем основную информацию датасета participants
participants.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB


In [16]:
#Выведем несколько строк для ознакомления
participants.head()

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac


In [17]:
#Выведем основную информацию датасета events
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 787286 entries, 0 to 787285
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   user_id     787286 non-null  object        
 1   event_dt    787286 non-null  datetime64[ns]
 2   event_name  787286 non-null  object        
 3   details     249022 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 24.0+ MB


In [18]:
#Выведем несколько строк для ознакомления
events.head()

,user_id,event_dt,event_name,details
0,GLOBAL,2020-12-01 00:00:00,End of Black Friday Ads Campaign,ZONE_CODE15
1,CCBE9E7E99F94A08,2020-12-01 00:00:11,registration,0.0
2,GLOBAL,2020-12-01 00:00:25,product_page,NaN
3,CCBE9E7E99F94A08,2020-12-01 00:00:33,login,NaN
4,CCBE9E7E99F94A08,2020-12-01 00:00:52,product_page,NaN


In [19]:
#Проверим данные на полные дубликаты
participants_duplicates = participants.duplicated().sum()
print(f"Количество полных дубликатов в датасете participants: {participants_duplicates}")

Количество полных дубликатов в датасете participants: 0


In [20]:
#Проверим данные на полные дубликаты
events_duplicates = events.duplicated().sum()
print(f"Количество полных дубликатов в датасете events: {events_duplicates}")

Количество полных дубликатов в датасете events: 36318


In [21]:
#Удалим полные дубликаты в events
events = events.drop_duplicates()
#Проверим результат
events_duplicates = events.duplicated().sum()
print(f"Количество полных дубликатов в датасете events: {events_duplicates}")

Количество полных дубликатов в датасете events: 0


## По таблице `participants` оценим корректность проведения теста:

   ### Выделим пользователей, участвующих в тесте, и проверим:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

In [22]:
other_test_users = participants[participants['ab_test'] != 'interface_eu_test']

#Отсеем только тех пользователей, кто участвует в тесте interface_eu_test и только в нем одном
interface_participants = participants[(participants['ab_test'] == 'interface_eu_test') & (~participants['user_id'].isin(other_test_users['user_id']))]

In [23]:
#Разделим пользователей по спискам
A_list = interface_participants[interface_participants['group'] == 'A']['user_id']
B_list = interface_participants[interface_participants['group'] == 'B']['user_id']
#Найдем пересечения в списках
intersection = list(set(A_list) & set(B_list))
#Выведем результат
len(intersection)

0

In [24]:
#Проверим равномерность распределения групп
pivot_group = pd.pivot_table(interface_participants, index='group', values='user_id', aggfunc='count').reset_index()
pivot_group

,group,user_id
0,A,4952
1,B,5011


In [25]:
# Рассчитаем количество пользователей в каждой группе
count_A = pivot_group[pivot_group['group'] == 'A']['user_id'].values[0]
count_B = pivot_group[pivot_group['group'] == 'B']['user_id'].values[0]

# Рассчитаем процентную разницу
percent_diff = abs(count_A - count_B) / count_A * 100

print(f"Процентная разница в количестве пользователей между группами A и B: {percent_diff:.2f}%")

Процентная разница в количестве пользователей между группами A и B: 1.19%


Приемлемая разница, группы распределены равномерно.

### Проанализируем данные о пользовательской активности по таблице `events`:

- оставим только события, связанные с участвующими в изучаемом тесте пользователями;

In [26]:
events_sorted = events[events['user_id'].isin(interface_participants['user_id'])]

- определим горизонт анализа: рассчитаем время (лайфтайм) совершения события пользователем после регистрации и оставим только те события, которые были выполнены в течение первых семи дней с момента регистрации;

In [27]:
#Для каждого user_id найдём время регистрации
reg_dates = (
    events_sorted[events_sorted['event_name'] == 'registration']
    .groupby('user_id')['event_dt']
    .min()
    .rename('reg_dt')
)
#Присоединим reg_dt ко всем строкам
events_sorted = events_sorted.merge(reg_dates, on='user_id', how='left')

In [28]:
#Посчитаем разницу в днях от регистрации
events_sorted['lifetime'] = events_sorted['event_dt'] - events_sorted['reg_dt']

#Отфильтруем события
events_sorted = events_sorted[(events_sorted['lifetime'] >= pd.Timedelta(days=0)) & (events_sorted['lifetime'] <= pd.Timedelta(days=7))]

In [29]:
#Проверим результат
events_sorted.lifetime.max()

Timedelta('6 days 23:58:10')

Оценим достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [30]:
# Задаём параметры
alpha = 0.05 # Уровень значимости
beta = 0.2  # Ошибка второго рода, часто 1 - мощность
power = 0.8  # Мощность теста
p = 0.3 # Базовый показатель конверсии
mde = 0.03  # Минимальный детектируемый эффект
effect_size = proportion_effectsize(p, p + mde)

# Инициализируем класс NormalIndPower
power_analysis = NormalIndPower()

# Рассчитываем размер выборки
sample_size = power_analysis.solve_power(
    effect_size = effect_size,
    power = power,
    alpha = alpha,
    ratio = 1 # Равномерное распределение выборок
)

print(f"Необходимый размер выборки для каждой группы: {int(sample_size)}")

Необходимый размер выборки для каждой группы: 3761


С помощью расчетов мы убедились в достаточности выборки для получения статистически значимых результатов A/B-теста.

- рассчитаем для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [31]:
#Считаем общее число участников в каждой группе
total_per_group = (
    interface_participants
    .groupby('group', as_index=False)
    .agg(total_visitors=('user_id', 'nunique'))
)

#Отбираем только тех пользователей, у кого есть хотя бы один purchase
purchasers = (
    events_sorted[events_sorted['event_name'] == 'purchase'][['user_id']]
        .merge(interface_participants[['user_id', 'group']], 
               on='user_id', 
               how='inner')
)
#Считаем количество пользователей по группам
purchasers_per_group = (
    purchasers
      .groupby('group', as_index=False)
      .agg(purchasers=('user_id', 'nunique'))
)

#Соединяем оба результата
result = (
    total_per_group
    .merge(purchasers_per_group, 
           on='group', 
           how='left')
)

# Рассчитаем долю покупателей
result['purchase_rate'] = round(result['purchasers'] / result['total_visitors'] * 100, 2)

In [32]:
result

,group,total_visitors,purchasers,purchase_rate
0,A,4952,1377,27.81
1,B,5011,1480,29.54


- сделаем предварительный общий вывод об изменении пользовательской активности в тестовой группе по сравнению с контрольной.

Предварительно, в группе с изменнёным интерфейсом сайта, мы можем увидеть увеличение конверсии в покупку по сравнению с контрольной группой, но окончательно подтвердить гипотезу мы сможем только проведя A/B-тестирование.

## Проведем оценку результатов A/B-тестирования:

- Проверим изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

Целевой метрикой явлется Конверсия в покупку.

Нулевая гипотеза: Упрощение интерфейса не приведёт к тому, что конверсия зарегистрированных пользователей в покупателей увеличится. Альтернативная гипотеза: Упрощение интерфейса приведёт к тому, что в течение семи дней после регистрации в системе конверсия зарегистрированных пользователей в покупателей увеличится

Для сравнения долей будем использовать Z-тест пропорций

In [33]:
#Размеры выборок
n_a = result.loc[result.group=='A', 'total_visitors'].iloc[0]
n_b = result.loc[result.group=='B', 'total_visitors'].iloc[0]

#Количество покупок
m_a = result.loc[result.group=='A', 'purchasers'].iloc[0]
m_b = result.loc[result.group=='B', 'purchasers'].iloc[0]

#Рассчитываем доли регистраций
p_a = m_a/n_a
p_b = m_b/n_b

In [34]:
#Выведем результат
print(f'n_a={n_a}, n_b={n_b}')
print(f'm_a={m_a}, m_b={m_b}')
print(f'p_a={p_a}, p_b={p_b}')

n_a=4952, n_b=5011
m_a=1377, m_b=1480
p_a=0.27806946688206785, p_b=0.29535022949511075


In [35]:
#Используем Z-тест пропорций
stat_ztest, p_value_ztest = proportions_ztest(
    [m_a, m_b],
    [n_a, n_b],
    alternative='smaller' 
)
print(p_value_ztest)

if p_value_ztest > alpha:
    print(f'pvalue={p_value_ztest} > {alpha}')
    print('Нулевая гипотеза находит подтверждение!')
else:
    print(f'pvalue={p_value_ztest} < {alpha}')
    print('Нулевая гипотеза не находит подтверждения!')

0.028262547212292124
pvalue=0.028262547212292124 < 0.05
Нулевая гипотеза не находит подтверждения!


**Выводы по результатам A/B-тестирования «упрощённого» интерфейса**


**Корректность проведения теста**

• В выборке участвовало 4 952 пользователя в контрольной (A) и 5 011 в тестовой (B) группах.

• Разница в размерах групп ≈1,2 % — укладывается в допустимые нормы (обычно до 5 %).

• Нет пересечений с конкурентными тестами, пользователей, одновременно в двух тестах, не обнаружено.

• Полных дубликатов по пользователям нет, пропусков не выявлено.



**Достаточность выборки**

• Базовая конверсия 30 %, мощность 80 %, α=0,05, MDE=3 % → требовалось по ~3 761 пользователя в каждой группе.

• Фактически в каждой группе >4 900 человек — выборка достаточна для обнаружения заявленного эффекта.



**Результаты по конверсии в покупку за первые 7 дней после регистрации**

• Контроль (A): 1 377 покупателей из 4 952 → 27,81 %.

• Тест (B): 1 480 покупателей из 5 011 → 29,54 %.

• Абсолютный приращение: +1,73 п.п. (относительный рост +6,2 %).



**Статистическая проверка**

• H0: Конверсия B ≤ конверсии A.

• H1: Конверсия B > конверсии A.

• Z-тест для разницы пропорций → p-value ≈0,028 < 0,05.

• При α=5 % нулевую гипотезу отвергаем → разница статистически значима.



**Итоговое заключение и рекомендация**

– Упрощённый интерфейс приводит к достоверному увеличению конверсии в покупку в течение недели после регистрации.

– Несмотря на то что фактический прирост (1,7 п.п.) чуть меньше изначально запрошенного MDE (3 п.п.), эффект статистически значим и коммерчески ощутим (6 % относительного роста продаж).

– Рекомендуется развёртывание нового интерфейса на всю аудиторию с одновременным мониторингом вторичных метрик (сессии, средний чек, удержание), чтобы убедиться в отсутствии побочных эффектов.